# Corrective RAG System
**Setup → Ingestion (upload, extract, clean, split) → Embeddings → ChromaDB → Retrieve → Grade chunks → Rewrite query & re-retrieve → Answer from verified context with sources → Unanswered questions → Evaluation → Gradio app (with document upload)**

Run the cells **top to bottom** in one Colab session.

| Requirement | Where |
|---|---|
| Prepare and process documents | Parts 1 to 5 (extract, clean, chunk) |
| Ingestion pipeline: upload, extract, clean, split | Parts 1 to 5 + Part 5 (`ingest_files`) + the **Upload** tab in the app |
| Embeddings + vector database | Part 6 (`multilingual-e5-small` + ChromaDB) |
| Retrieve and evaluate chunks | Part 2 |
| Rewrite query and re-retrieve | Part 3 |
| Answer from verified context only, with sources | Part 4 |
| Handle unanswered questions | Part 4 (`not_found` + `NOT_ENOUGH_INFO`) and the evaluation |
| Deploy with Gradio | Last part |

## 1. Setup

In [1]:
!pip -q install pypdf python-docx python-pptx chromadb sentence-transformers langchain-text-splitters pandas gradio groq

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


## 2. Unzip + course names
Unzips `data.zip` (if needed) and renames `course1/2/3` to the real course codes. Safe to re-run.

In [2]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/data.zip")
DATA_DIR = Path("/content/data")

if ZIP_PATH.exists() and not DATA_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("/content/")

# if the zip had the course folders at its root, they are in /content/ directly
if not DATA_DIR.exists():
    DATA_DIR.mkdir()
    for name in ["course1", "course2", "course3", "M269", "TM112", "TM275"]:
        if Path("/content", name).exists():
            Path("/content", name).rename(DATA_DIR / name)

RENAME = {"course1": "TM112", "course2": "TM275", "course3": "M269"}
for old, new in RENAME.items():
    if (DATA_DIR / old).exists() and not (DATA_DIR / new).exists():
        (DATA_DIR / old).rename(DATA_DIR / new)

SUPPORTED = {".pdf", ".docx", ".pptx", ".txt", ".csv"}
for course in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    files = [f.name for f in course.iterdir() if f.suffix.lower() in SUPPORTED]
    print(course.name, "->", len(files), "files")
    for f in files: print("   ", f)

M269 -> 5 files
    100-M269_MTA_REVISION.pdf
    M269_lec2.pptx
    M269_lec2_slides.csv
    M269_lec2_notes.txt
    m269.docx
TM112 -> 5 files
    TM112 Meeting12-Cryptography.pptx
    Meeting 6 جاهز للطباعة.pdf
    TM112 Meeting12-Cryptography_slides.csv
    TM112 Meeting12-Cryptography_notes.txt
    Moc TM112 answerkey.docx
TM275 -> 5 files
    MTA Focus exam (1).docx
    TM275 - Meeting 3# Distributed Systems (Theoretical + Docker).pdf
    TM275 - Meeting 6# AWS Cloud (2).pptx
    TM275 - Meeting 6# AWS Cloud (2)_slides.csv
    TM275 - Meeting 6# AWS Cloud (2)_notes.txt


## 3. Extraction
Each loader returns `(text, location)` so every chunk can be cited with its **file + page/slide**.
`_notes.txt` and `_slides.csv` files are skipped because they are just exports of the `.pptx`.

In [3]:
import pandas as pd
from pypdf import PdfReader
from docx import Document
from pptx import Presentation

def load_pdf(p):
    out = []
    for i, page in enumerate(PdfReader(str(p)).pages, 1):
        t = page.extract_text() or ""
        if t.strip():
            out.append((t, f"page {i}"))
    return out

def load_docx(p):
    d = Document(str(p))
    parts = [para.text for para in d.paragraphs if para.text.strip()]
    for tbl in d.tables:
        for row in tbl.rows:
            parts.append(" | ".join(c.text.strip() for c in row.cells))
    return [("\n".join(parts), "document")]

def load_pptx(p):
    out = []
    for i, slide in enumerate(Presentation(str(p)).slides, 1):
        texts = []
        for sh in slide.shapes:
            if sh.has_text_frame:
                texts.append(sh.text_frame.text)
            if getattr(sh, "has_table", False) and sh.has_table:
                for r in sh.table.rows:
                    texts.append(" | ".join(c.text for c in r.cells))
        if slide.has_notes_slide:
            texts.append(slide.notes_slide.notes_text_frame.text)
        t = "\n".join(x for x in texts if x.strip())
        if t.strip():
            out.append((t, f"slide {i}"))
    return out

def load_txt(p):
    return [(Path(p).read_text(encoding="utf-8", errors="ignore"), "text file")]

def load_csv(p):
    df = pd.read_csv(p, encoding="utf-8", on_bad_lines="skip").fillna("")
    rows = ["; ".join(f"{c}: {v}" for c, v in r.items() if str(v).strip()) for _, r in df.iterrows()]
    return [("\n".join(rows), "csv")]

LOADERS = {".pdf": load_pdf, ".docx": load_docx, ".pptx": load_pptx, ".txt": load_txt, ".csv": load_csv}

raw_docs = []   # {"text", "course", "source", "loc"}
for course in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    for f in sorted(course.iterdir()):
        if f.stem.endswith(("_notes", "_slides")):   # derived from the .pptx, skip duplicates
            continue
        if f.suffix.lower() == ".csv" and f.stat().st_size > 2_000_000:   # big table -> analyze_data tool, not RAG
            print("Skipping large CSV (used by the analyze_data tool):", f.name)
            continue
        loader = LOADERS.get(f.suffix.lower())
        if not loader:
            continue
        try:
            for text, loc in loader(f):
                raw_docs.append({"text": text, "course": course.name, "source": f.name, "loc": loc})
        except Exception as e:
            print("FAILED:", f.name, "->", e)

print("Extracted units:", len(raw_docs))
print(pd.DataFrame(raw_docs).groupby(["course", "source"]).size())

Extracted units: 253
course  source                                                           
M269    100-M269_MTA_REVISION.pdf                                            14
        M269_lec2.pptx                                                       46
        m269.docx                                                             1
TM112   Meeting 6 جاهز للطباعة.pdf                                            9
        Moc TM112 answerkey.docx                                              1
        TM112 Meeting12-Cryptography.pptx                                    40
TM275   MTA Focus exam (1).docx                                               1
        TM275 - Meeting 3# Distributed Systems (Theoretical + Docker).pdf    42
        TM275 - Meeting 6# AWS Cloud (2).pptx                                99
dtype: int64


## 4. Cleaning

In [4]:
import re

def fix_doubled_chars(t: str) -> str:
    """Some PDFs render bold text twice (e.g. 'AAllggoorriitthhmmss'). Collapse words made only of doubled letters."""
    def fix(m):
        w = m.group(0)
        if (len(w) >= 4 and len(w) % 2 == 0 and any(ch.isalpha() for ch in w)
                and all(w[i] == w[i + 1] for i in range(0, len(w), 2))):
            return w[::2]
        return w
    return re.sub(r"\w+", fix, t)

def clean_text(t: str) -> str:
    t = fix_doubled_chars(t)
    t = t.replace("\x00", " ")
    t = re.sub(r"(?im)^\s*(page\s*)?\d{1,3}\s*$", "", t)   # lone page numbers
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

for d in raw_docs:
    d["text"] = clean_text(d["text"])
raw_docs = [d for d in raw_docs if len(d["text"]) > 30]
print("After cleaning:", len(raw_docs), "units")

After cleaning: 243 units


## 5. Chunking
~900 characters, 150 overlap, split at paragraph/sentence boundaries (Arabic punctuation included).

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import hashlib

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900, chunk_overlap=150,
    separators=["\n\n", "\n", ". ", "؟ ", "، ", " ", ""],
)

chunks, seen = [], set()
for d in raw_docs:
    for piece in splitter.split_text(d["text"]):
        piece = piece.strip()
        if len(piece) < 40:
            continue
        h = hashlib.md5((d["course"] + re.sub(r"\s+", " ", piece.lower())).encode()).hexdigest()
        if h in seen:
            continue
        seen.add(h)
        chunks.append({"id": h, "text": piece, "course": d["course"], "source": d["source"], "loc": d["loc"]})

print("Total chunks:", len(chunks))
print(pd.DataFrame(chunks).groupby("course").size())
print("\nSample chunk:\n", chunks[0])

Total chunks: 296
course
M269      79
TM112     89
TM275    128
dtype: int64

Sample chunk:
 {'id': '8b7b9943f68892f4ccabd9d26a4d86a2', 'text': '1 M269 – Algorithm and Data structure – MTA Revision \n \n \n \n \nAlgorithms && Data Structure Using python \nM269 MTA sheet', 'course': 'M269', 'source': '100-M269_MTA_REVISION.pdf', 'loc': 'page 1'}


## 6. Embeddings + Vector DB
`intfloat/multilingual-e5-small` (Arabic + English, uses `passage:` / `query:` prefixes) and ChromaDB with cosine similarity.

In [6]:
from sentence_transformers import SentenceTransformer
import chromadb

embedder = SentenceTransformer("intfloat/multilingual-e5-small")

client = chromadb.PersistentClient(path="chroma_db")
try:
    client.delete_collection("course_docs")
except Exception:
    pass
collection = client.create_collection("course_docs", metadata={"hnsw:space": "cosine"})

BATCH = 128
for i in range(0, len(chunks), BATCH):
    batch = chunks[i:i + BATCH]
    embs = embedder.encode(["passage: " + c["text"] for c in batch],
                           normalize_embeddings=True, show_progress_bar=False).tolist()
    collection.add(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        embeddings=embs,
        metadatas=[{"course": c["course"], "source": c["source"], "loc": c["loc"]} for c in batch],
    )
print("Stored in Chroma:", collection.count())

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Stored in Chroma: 296


## 7. Retrieval test

In [7]:
def retrieve(query: str, k: int = 5, course: str | None = None):
    q = embedder.encode(["query: " + query], normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=q, n_results=k,
                           where={"course": course} if course else None)
    return [
        {"text": t, "meta": m, "score": 1 - d}     # cosine similarity
        for t, m, d in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])
    ]

for query in ["What is the internet?", "What is symmetric encryption?"]:
    print("Q:", query)
    for r in retrieve(query, k=3):
        print(f"  {r['score']:.3f} | {r['meta']['course']} | {r['meta']['source']} | {r['meta']['loc']}")
        print("   ", r["text"][:200].replace("\n", " "))
    print()

Q: What is the internet?
  0.875 | TM112 | Meeting 6 جاهز للطباعة.pdf | page 6
    • Lots of different networks of different sizes and shapes, all linked together to form the internet.  • When a network (LAN) joins to other networks (i.e., becomes part of the internet) – it has to c
  0.865 | TM112 | Meeting 6 جاهز للطباعة.pdf | page 1
    Ahmed  Elbhbaty   Meeting #6 Network technologies | The Internet    ➢ Network of networks    • The term internet derives from that the internet is itself made up of many other joined-together networks
  0.864 | TM112 | Meeting 6 جاهز للطباعة.pdf | page 4
    → Internet (IP) addresses: these have provided a universal addressing scheme.  → Protocols: these have allowed many new services to be added to existing architecture.  • It is the technology that unde

Q: What is symmetric encryption?
  0.887 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 32
    It is well-suited to encrypting any amount of data. Unlike asymmetric encryption, symmetric encry

# Part 2: Evaluate the retrieved chunks

For every question: retrieve top-k, an LLM grades each chunk (`relevant` / `ambiguous` / `irrelevant`), then the system decides:
- `correct`: at least one relevant chunk, keep only the relevant ones
- `ambiguous`: no clearly relevant chunk but some partial ones, so rewrite the query (Part 3)
- `incorrect`: nothing useful, so rewrite, and if still weak answer "not found"

### LLM setup (Groq)
Add `GROQ_API_KEY` in Colab Secrets, or paste it when asked. All LLM calls go through `call_llm`, so switching to Gemini later means editing one function.

Note: Groq retired `llama-3.3-70b-versatile` on Aug 16, 2026, so the notebook auto-picks `openai/gpt-oss-120b` (Groq's recommended replacement).

In [8]:
import os, json, time
from getpass import getpass
from groq import Groq

def _get_groq_key():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("GROQ_API_KEY")          # Colab Secrets (name must be exactly GROQ_API_KEY, Notebook access ON)
    except Exception:
        key = None
    if not key:
        key = getpass("Groq API key (starts with gsk_): ")
    return key.strip().strip('"').strip("'")         # remove accidental spaces / quotes

os.environ["GROQ_API_KEY"] = _get_groq_key()
if not os.environ["GROQ_API_KEY"].startswith("gsk_"):
    raise ValueError("That does not look like a Groq key (it must start with gsk_). Create one at https://console.groq.com/keys")
llm_client = Groq()

# Groq retired llama-3.3-70b-versatile (Aug 2026). Pick the first preferred model your key can access.
PREFERRED = ["openai/gpt-oss-120b", "qwen/qwen3.8-27b", "openai/gpt-oss-20b"]
try:
    available = {m.id for m in llm_client.models.list().data}
except Exception as e:
    if "401" in str(e) or "Invalid API Key" in str(e):
        raise SystemExit("Groq rejected the key (401 Invalid API Key). Create a NEW key at console.groq.com/keys, update the GROQ_API_KEY secret, then Runtime > Restart session and run again.")
    raise
MODEL = next((m for m in PREFERRED if m in available), None)
if MODEL is None:
    print("None of the preferred models found. Available models:")
    for m in sorted(available): print("  ", m)
    raise SystemExit("Set MODEL manually to one of the text models above.")
print("Using model:", MODEL)

def call_llm(system: str, user: str, json_mode: bool = False,
             temperature: float = 0.0, max_tokens: int = 2000, retries: int = 3) -> str:
    kwargs = {"response_format": {"type": "json_object"}} if json_mode else {}
    if "gpt-oss" in MODEL:
        kwargs["reasoning_effort"] = "low"     # reasoning model: keep it fast, leave room for the answer
    for attempt in range(retries):
        try:
            r = llm_client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}],
                temperature=temperature, max_tokens=max_tokens, **kwargs)
            return r.choices[0].message.content
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"LLM error ({e.__class__.__name__}: {str(e)[:120]}), retrying...")
            time.sleep(2 * (attempt + 1))

print(call_llm("You are helpful.", "Say OK."))

Groq API key (starts with gsk_): ··········
Using model: openai/gpt-oss-120b
OK.


### Robust LLM calls
Handles the occasional empty-JSON error of reasoning models and **switches to a backup model automatically when Groq's rate limit is hit**.

In [9]:
# ------------------------------------------------------------------
# Robust LLM wrapper
#  1) Reasoning models (gpt-oss) sometimes spend all their tokens thinking and return an empty JSON
#     -> Groq error "json_validate_failed": give JSON calls more room and fall back to plain text.
#  2) On a rate limit (429) switch to the next available model instead of failing / waiting.
# ------------------------------------------------------------------
if "_base_call_llm" not in globals():
    _base_call_llm = call_llm

def _rotate_model() -> bool:
    global MODEL
    pool = [m for m in PREFERRED if m in available and m != MODEL]
    if not pool:
        return False
    MODEL = pool[0]
    print(f"Rate limit reached: switched to {MODEL}")
    return True

def _call_once(system, user, json_mode, temperature, max_tokens, retries):
    if not json_mode:
        return _base_call_llm(system, user, False, temperature, max_tokens, retries)
    big = max(max_tokens, 2500)
    try:
        return _base_call_llm(system, user, True, temperature, big, 1)
    except Exception as e:
        if "json_validate_failed" not in str(e) and "400" not in str(e):      # e.g. rate limit -> normal retries
            return _base_call_llm(system, user, True, temperature, big, retries)
    return _base_call_llm(system + "\nReply ONLY with one valid JSON object and nothing else.", user,
                          False, temperature, max(max_tokens, 3500), retries)

def call_llm(system: str, user: str, json_mode: bool = False,
             temperature: float = 0.0, max_tokens: int = 2000, retries: int = 3) -> str:
    for attempt in range(2):                       # the 2nd pass runs after a model switch
        try:
            return _call_once(system, user, json_mode, temperature, max_tokens, retries)
        except Exception as e:
            is_rate = "429" in str(e) or "rate_limit" in str(e).lower()
            if is_rate and attempt == 0 and _rotate_model():
                continue
            raise

# Arabic question -> Arabic "not found" message
NOT_FOUND_AR = "عذرًا، ليس لدي أي معلومات عن هذا السؤال في المستندات."

COURSES = sorted({c["course"] for c in chunks})

### Chunk grader
One LLM call grades all k chunks together (saves API calls).

In [10]:
GRADER_SYSTEM = """You are a relevance grader for a document question-answering system.
Given a question and numbered text chunks, judge whether each chunk contains information that helps answer the question.
Verdicts:
- "relevant": the chunk contains information that directly helps answer the question. A formal one-line definition is NOT required: explanations, properties, purposes, examples or key facts that together answer the question count as relevant.
- "ambiguous": the chunk is about a related topic but only touches the question tangentially, or the key information is missing.
- "irrelevant": a different topic, or only titles / table of contents / headings with no explanatory content.
Do not rely on keyword overlap alone; judge by meaning.
- For "What is X?" / "Explain X" / "ما هو X" questions, a chunk that explains how X works, its properties, purpose, advantages or usage IS relevant (score 0.7 or more) even if it never gives a one-line definition.
- Never penalize a chunk because some information is missing from it: other chunks may supply it. Use "ambiguous" only when the chunk is about a different but related topic.
The question and chunks may be in Arabic or English.
Scores: relevant >= 0.7, ambiguous 0.3-0.6, irrelevant < 0.3.
Reply ONLY with JSON."""

REL_THRESHOLD = 0.6   # score needed to count a chunk as relevant
RESCUE_GRADE = 0.5    # an 'ambiguous' chunk with at least this grade ...
RESCUE_SIM   = 0.86   # ... AND at least this embedding similarity is kept as relevant (the grader can be too strict)

def grade_chunks(question: str, chunks: list[dict]) -> list[dict]:
    numbered = "\n\n".join(f"[{i}] {c['text'][:800]}" for i, c in enumerate(chunks))
    user = (f"Question: {question}\n\nChunks:\n{numbered}\n\n"
            'Return JSON: {"grades": [{"index": 0, "verdict": "relevant|ambiguous|irrelevant", '
            '"score": 0.0, "reason": "one short sentence"}]} with one entry per chunk.')
    raw = call_llm(GRADER_SYSTEM, user, json_mode=True)
    try:
        items = json.loads(raw).get("grades", [])
    except Exception:
        items = []
    by_idx = {int(g.get("index", -1)): g for g in items if isinstance(g, dict)}

    graded = []
    for i, c in enumerate(chunks):
        g = by_idx.get(i, {})
        verdict = g.get("verdict", "irrelevant")
        try:
            score = float(g.get("score", 0.0))
        except (TypeError, ValueError):
            score = 0.0
        if verdict not in {"relevant", "ambiguous", "irrelevant"}:
            verdict = "irrelevant"
        if verdict == "relevant" and score < REL_THRESHOLD:
            verdict = "ambiguous"
        if verdict == "ambiguous" and score >= RESCUE_GRADE and c["score"] >= RESCUE_SIM:
            verdict = "relevant"      # rescue: partially relevant and very similar to the question
        graded.append({**c, "verdict": verdict, "grade": score, "reason": g.get("reason", "")})
    return graded

### Retrieval status

In [11]:
def evaluate_retrieval(question: str, k: int = 5, course: str | None = None,
                       search_query: str | None = None) -> dict:
    # retrieve with the (possibly rewritten) search query, but always grade against the ORIGINAL question
    chunks = retrieve(search_query or question, k=k, course=course)
    graded = grade_chunks(question, chunks)

    relevant  = [g for g in graded if g["verdict"] == "relevant"]
    ambiguous = [g for g in graded if g["verdict"] == "ambiguous"]

    if relevant:
        status, kept = "correct", relevant
    elif ambiguous:
        status, kept = "ambiguous", []
    else:
        status, kept = "incorrect", []

    return {"question": question, "status": status,
            "kept_chunks": kept, "graded": graded,
            "confidence": max((g["grade"] for g in graded), default=0.0)}

def show(result: dict):
    print(f"Q: {result['question']}")
    print(f"Status: {result['status'].upper()}  |  kept: {len(result['kept_chunks'])}  |  best grade: {result['confidence']:.2f}")
    for g in result["graded"]:
        m = g["meta"]
        print(f"  [{g['verdict']:<10}] grade={g['grade']:.2f} sim={g['score']:.2f} | {m['course']} | {m['source']} | {m['loc']}")
        print(f"      reason: {g['reason']}")
    print()

### Test it
Expected: in-domain → `correct`, Arabic → `correct`, out-of-domain → `incorrect`.

In [12]:
tests = [
    "What is symmetric encryption?",
    "What is the internet?",
    "ما هو التشفير المتماثل؟",
    "Who won the 2022 football World Cup?",     # out of domain on purpose
]
results = []
for q in tests:
    res = evaluate_retrieval(q, k=5)
    show(res)
    results.append(res)
    time.sleep(1)

pd.DataFrame([{"question": r["question"], "status": r["status"],
               "kept": len(r["kept_chunks"]), "best_grade": round(r["confidence"], 2)} for r in results])

Q: What is symmetric encryption?
Status: CORRECT  |  kept: 3  |  best grade: 0.82
  [relevant  ] grade=0.82 sim=0.89 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 32
      reason: Explains properties and use cases of symmetric encryption, directly answering the question.
  [relevant  ] grade=0.81 sim=0.89 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 21
      reason: Describes symmetric encryption speed and common usage, providing relevant information.
  [relevant  ] grade=0.78 sim=0.87 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 32
      reason: Compares symmetric and asymmetric encryption and lists reasons why symmetric is used, directly relevant.
  [irrelevant] grade=0.12 sim=0.86 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 2
      reason: Only a table of contents with no explanatory content about symmetric encryption.
  [irrelevant] grade=0.10 sim=0.85 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 33
      reason: Focuses solely on asymmetric crypt

,question,status,kept,best_grade
0,What is symmetric encryption?,correct,3,0.82
1,What is the internet?,correct,5,0.90
2,ما هو التشفير المتماثل؟,correct,1,0.70
3,Who won the 2022 football World Cup?,incorrect,0,0.00


# Part 3: Correction (rewrite the query and re-retrieve)

If the first retrieval has no `relevant` chunk:
1. The LLM **rewrites** the question into a better search query (it also translates Arabic questions to English, because the course material is in English), using the grader's notes about what was missing.
2. We retrieve again and grade again (up to `MAX_REWRITES` times).
3. Final status:
   - `verified`: at least one relevant chunk was found, so the answer will use only these.
   - `partial`: only ambiguous chunks, so the answer will be flagged as low confidence.
   - `not_found`: nothing useful, so the system will say it could not find the answer.

In [13]:
REWRITE_SYSTEM = """You rewrite user questions into better search queries for a vector search over university course material (computing, networks, cryptography, cloud computing, algorithms and data structures).
Rules:
- Output ONE standalone query in English. If the question is in Arabic, translate it faithfully.
- Keep the SAME meaning and make it short (3 to 8 words) using the core technical terms.
- Do NOT add words like "definition", "examples", "lecture" or "slide" unless the question asks for them.
- Use the evaluator notes only to choose better technical terms or synonyms, and do NOT repeat a previous query.
- Ignore evaluator notes that complain about a missing definition: keep the query about the topic itself.
Reply ONLY with JSON: {"query": "..."}"""

def rewrite_query(question: str, previous_queries: list[str], notes: str) -> str:
    user = (f"Original question: {question}\n"
            f"Previous queries (did not work): {previous_queries}\n"
            f"Evaluator notes about what was retrieved: {notes}")
    raw = call_llm(REWRITE_SYSTEM, user, json_mode=True, max_tokens=800)
    try:
        q = json.loads(raw).get("query", "").strip()
    except Exception:
        q = ""
    return q or question

In [14]:
MAX_REWRITES = 2

def _key(g):
    return (g["meta"]["source"], g["meta"]["loc"], g["text"][:80])

def corrective_retrieve(question: str, k: int = 5, course: str | None = None,
                        max_rewrites: int = MAX_REWRITES) -> dict:
    trace, tried = [], [question]
    relevant, partial = {}, {}

    def absorb(res):
        for g in res["graded"]:
            key = _key(g)
            if g["verdict"] == "relevant":
                relevant[key] = g
            elif g["verdict"] == "ambiguous" and key not in relevant:
                partial[key] = g

    res = evaluate_retrieval(question, k=k, course=course)
    trace.append({"query": question, "status": res["status"]})
    absorb(res)

    rewrites = 0
    while not relevant and rewrites < max_rewrites:
        notes = " | ".join(g["reason"] for g in res["graded"][:4] if g["reason"])
        new_q = rewrite_query(question, tried, notes)
        if new_q in tried:
            break
        rewrites += 1
        tried.append(new_q)
        res = evaluate_retrieval(question, k=k, course=course, search_query=new_q)
        trace.append({"query": new_q, "status": res["status"]})
        absorb(res)

    for key in list(partial):
        if key in relevant:
            del partial[key]

    if relevant:
        final, context = "verified", sorted(relevant.values(), key=lambda g: -g["grade"])
    elif partial:
        final, context = "partial", sorted(partial.values(), key=lambda g: -g["grade"])[:3]
    else:
        final, context = "not_found", []

    return {"question": question, "final_status": final, "context": context,
            "trace": trace, "rewrites": rewrites}

def show_corrective(r: dict):
    print(f"Q: {r['question']}")
    for i, t in enumerate(r["trace"]):
        label = "original" if i == 0 else f"rewrite {i}"
        print(f"  [{label}] {t['status']:<10} <- {t['query']}")
    print(f"  FINAL: {r['final_status'].upper()}  |  context chunks: {len(r['context'])}")
    for g in r["context"]:
        m = g["meta"]
        print(f"     - {g['verdict']:<9} {g['grade']:.2f} | {m['course']} | {m['source']} | {m['loc']}")
    print()

### Test the full correction loop
Expected: the Arabic question gets rewritten to English and becomes `verified`; the World Cup question ends as `not_found`.

In [15]:
corrective_results = []
for q in tests:
    r = corrective_retrieve(q, k=5)
    show_corrective(r)
    corrective_results.append(r)
    time.sleep(1)

pd.DataFrame([{"question": r["question"], "final": r["final_status"],
               "rewrites": r["rewrites"], "context": len(r["context"])} for r in corrective_results])

Q: What is symmetric encryption?
  [original] correct    <- What is symmetric encryption?
  FINAL: VERIFIED  |  context chunks: 3
     - relevant  0.90 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 21
     - relevant  0.90 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 32
     - relevant  0.80 | TM112 | TM112 Meeting12-Cryptography.pptx | slide 32

Q: What is the internet?
  [original] correct    <- What is the internet?
  FINAL: VERIFIED  |  context chunks: 5
     - relevant  0.90 | TM112 | Meeting 6 جاهز للطباعة.pdf | page 6
     - relevant  0.90 | TM112 | Meeting 6 جاهز للطباعة.pdf | page 1
     - relevant  0.80 | TM112 | Meeting 6 جاهز للطباعة.pdf | page 4
     - relevant  0.50 | TM112 | Meeting 6 جاهز للطباعة.pdf | page 8
     - relevant  0.50 | TM112 | Meeting 6 جاهز للطباعة.pdf | page 8

Q: ما هو التشفير المتماثل؟
  [original] correct    <- ما هو التشفير المتماثل؟
  FINAL: VERIFIED  |  context chunks: 1
     - relevant  0.70 | TM112 | TM112 Meeting12-Cryptography.pptx

,question,final,rewrites,context
0,What is symmetric encryption?,verified,0,3
1,What is the internet?,verified,0,5
2,ما هو التشفير المتماثل؟,verified,0,1
3,Who won the 2022 football World Cup?,not_found,2,0


# Part 4: Answer generation (verified context only, with sources)

- `verified`: answer only from the relevant chunks, with inline citations `[1]`, `[2]`.
- `partial`: same, but the answer starts with a low-confidence warning.
- `not_found` (or the model says the context is not enough): a helpful **scope message** (what the documents cover + example questions), **no guessing**.
- The answer language follows the question (Arabic question → Arabic answer).
- Sources shown are the chunks that were actually cited: course, file, page/slide.

In [16]:
ARABIC_RE = re.compile(r"[\u0600-\u06FF]")

LOW_CONF_EN  = "⚠️ Low confidence: the documents only partly cover this question."
LOW_CONF_AR  = "⚠️ ثقة منخفضة: المستندات تغطي هذا السؤال جزئياً فقط."

ANSWER_SYSTEM = """You answer questions about university course material using ONLY the numbered context chunks provided.
Rules:
- Use only facts that appear in the context. Never add outside knowledge, even if you know the answer.
- Cite the chunks you used inline like [1] or [2][3] right after each statement.
- Answer in the same language as the question. If the question is Arabic, answer in Arabic and keep technical terms in English in parentheses where helpful.
- Be clear and concise (a short paragraph or a few bullets).
- If the context only partly answers the question, answer the part it supports and say clearly which part is not covered.
- Reply with exactly NOT_ENOUGH_INFO only when the context contains nothing relevant to the question."""

def build_context(chunks: list[dict]) -> str:
    return "\n\n".join(
        f"[{i}] ({g['meta']['course']} | {g['meta']['source']} | {g['meta']['loc']})\n{g['text']}"
        for i, g in enumerate(chunks, 1))

# ------------------------------------------------------------------
# Out-of-scope questions: no "I don't know" reply. Say what the documents cover and suggest questions they can answer.
# ------------------------------------------------------------------
_EXAMPLES_CACHE = {}

def kb_example_questions(arabic: bool, n: int = 4) -> list:
    key = (len(chunks), arabic)
    if key in _EXAMPLES_CACHE:
        return _EXAMPLES_CACHE[key]
    import random
    pool = random.Random(0).sample(chunks, min(14, len(chunks)))
    snippets = "\n".join(f"- ({c['course']}) {c['text'][:150]}".replace("\n", " ") for c in pool)
    qs = []
    try:
        raw = call_llm(
            f"You write short example questions a student could ask about the given course material. "
            f"Write {n} varied questions (different courses/topics), each under 12 words, "
            f"in {'Arabic' if arabic else 'English'}. Reply ONLY with JSON: {{\"questions\": [\"...\"]}}",
            "Material:\n" + snippets, json_mode=True, max_tokens=600)
        m = re.search(r"\{.*\}", raw or "", re.S)
        qs = [str(x).strip() for x in json.loads(m.group(0)).get("questions", []) if str(x).strip()][:n]
    except Exception:
        qs = []
    if not qs:   # fallback: use the document names as topics
        qs = sorted({Path(c["source"]).stem for c in chunks})[:n]
    _EXAMPLES_CACHE[key] = qs
    return qs

def scope_reply(question: str, is_ar: bool) -> str:
    courses = ", ".join(sorted({c["course"] for c in chunks}))
    qs = "\n".join(f"- {q}" for q in kb_example_questions(is_ar))
    if is_ar:
        return (f"📚 هذا السؤال خارج نطاق مواد الكورسات المتاحة ({courses}).\n\n"
                f"أقدر أجاوب بدقة من المستندات على أسئلة مثل:\n{qs}")
    return (f"📚 This question is outside the course material available here ({courses}).\n\n"
            f"I can answer from the documents with sources, for example:\n{qs}")

def _norm_cites(text: str) -> str:
    # gpt-oss sometimes writes citations as 【1†L1-L4】: normalise every variant to plain [1]
    text = re.sub(r"[【\[]\s*(\d+)\s*†[^】\]]*[】\]]", r"[\1]", text)
    return text.replace("【", "[").replace("】", "]")

def answer_question(question: str, k: int = 5, course: str | None = None) -> dict:
    is_ar = bool(ARABIC_RE.search(question))
    not_found = lambda: scope_reply(question, is_ar)
    r = corrective_retrieve(question, k=k, course=course)

    if r["final_status"] == "not_found":
        return {"question": question, "status": "not_found", "answer": not_found(),
                "sources": [], "trace": r["trace"]}

    ctx = r["context"]
    user = f"Question: {question}\n\nContext:\n{build_context(ctx)}"
    answer = _norm_cites(call_llm(ANSWER_SYSTEM, user, max_tokens=2000).strip())

    # The grader already verified these chunks as relevant, so "NOT_ENOUGH_INFO" is suspicious
    # (typically an Arabic question over English chunks). Retry once with an explicit instruction.
    if ("NOT_ENOUGH_INFO" in answer or not answer) and r["final_status"] == "verified":
        retry_system = ANSWER_SYSTEM + (
            "\n- IMPORTANT: these chunks were already verified as relevant to the question. The question may be "
            "in a different language than the chunks: translate as needed, answer from the chunks in the "
            "question's language, and do NOT reply NOT_ENOUGH_INFO.")
        answer = _norm_cites(call_llm(retry_system, user, max_tokens=2500).strip())

    if "NOT_ENOUGH_INFO" in answer:
        return {"question": question, "status": "not_found", "answer": not_found(),
                "sources": [], "trace": r["trace"]}

    cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer) if 1 <= int(n) <= len(ctx)})
    used = [ctx[i - 1] for i in cited] or ctx
    sources, seen = [], set()          # one entry per (file, page/slide), keeping the best grade
    for g in sorted(used, key=lambda g: -g["grade"]):
        key = (g["meta"]["source"], g["meta"]["loc"])
        if key in seen:
            continue
        seen.add(key)
        sources.append({"course": g["meta"]["course"], "source": g["meta"]["source"],
                        "loc": g["meta"]["loc"], "relevance": round(g["grade"], 2)})

    if r["final_status"] == "partial":
        answer = (LOW_CONF_AR if is_ar else LOW_CONF_EN) + "\n\n" + answer
    return {"question": question, "status": r["final_status"], "answer": answer,
            "sources": sources, "trace": r["trace"]}

def show_answer(res: dict):
    print(f"Q: {res['question']}")
    print(f"Status: {res['status'].upper()}  |  searches: {[t['query'] for t in res['trace']]}")
    print("\nAnswer:\n" + res["answer"])
    if res["sources"]:
        print("\nSources:")
        for s_ in res["sources"]:
            print(f"  - {s_['course']} | {s_['source']} | {s_['loc']}  (relevance {s_['relevance']})")
    print("\n" + "-" * 70 + "\n")

### Test it
Expected: grounded answers with sources for in-domain questions, an Arabic answer for the Arabic question, and the scope message for the World Cup question.

In [17]:
answer_tests = tests + ["What is Docker?"]
answers = []
for q in answer_tests:
    res = answer_question(q)
    show_answer(res)
    answers.append(res)
    time.sleep(1)

pd.DataFrame([{"question": a["question"], "status": a["status"], "sources": len(a["sources"])} for a in answers])

Q: What is symmetric encryption?
Status: VERIFIED  |  searches: ['What is symmetric encryption?']

Answer:
Symmetric encryption is a highly secure and fast form of encryption that uses small keys, making it suitable for encrypting large or unknown amounts of data [1][2][3]. It is widely used because modern CPUs can perform it in hardware or software at high speeds, even on modest computer equipment [1][2]. Unlike asymmetric encryption, symmetric encryption can handle data where the final size is not known in advance, such as live internet telephone calls [3]. Consequently, most encrypted data sent over networks relies on symmetric algorithms [1].

Sources:
  - TM112 | TM112 Meeting12-Cryptography.pptx | slide 21  (relevance 0.9)
  - TM112 | TM112 Meeting12-Cryptography.pptx | slide 32  (relevance 0.9)

----------------------------------------------------------------------

Q: What is the internet?
Status: VERIFIED  |  searches: ['What is the internet?']

Answer:
The internet is defined

,question,status,sources
0,What is symmetric encryption?,verified,2
1,What is the internet?,verified,4
2,ما هو التشفير المتماثل؟,verified,1
3,Who won the 2022 football World Cup?,not_found,0
4,What is Docker?,verified,2


# Part 5: Ingestion of newly uploaded documents
`ingest_files` runs the **same pipeline** on any new file (extract → clean → split → de-duplicate → embed → store in ChromaDB), so users can add documents without rebuilding the index. This function is what the **Upload** tab in the app calls.
Big CSV tables (> 2 MB) are skipped: they are tables, not text.

In [18]:
import hashlib
from pathlib import Path

UPLOAD_DIR = Path("/content/uploads"); UPLOAD_DIR.mkdir(exist_ok=True)

def ingest_files(file_paths, course: str = "UPLOADS"):
    """Extract -> clean -> split -> de-duplicate -> embed -> store, for new files. Returns (n_new_chunks, report_lines)."""
    course = (course or "UPLOADS").strip().upper()
    total, report = 0, []
    for fp in file_paths:
        f = Path(fp)
        loader = LOADERS.get(f.suffix.lower())
        if loader is None:
            report.append(f"⚠️ {f.name}: unsupported file type (use pdf, docx, pptx, txt, csv)")
            continue
        if f.suffix.lower() == ".csv" and f.stat().st_size > 2_000_000:
            report.append(f"⚠️ {f.name}: large table skipped (tables are not text chunks)")
            continue
        try:
            units = loader(f)
        except Exception as e:
            report.append(f"❌ {f.name}: could not read ({e.__class__.__name__})")
            continue
        new = []
        for text, loc in units:
            text = clean_text(text)
            if len(text) <= 30:
                continue
            for piece in splitter.split_text(text):
                piece = piece.strip()
                if len(piece) < 40:
                    continue
                h = hashlib.md5((course + re.sub(r"\s+", " ", piece.lower())).encode()).hexdigest()
                if h in seen:
                    continue
                seen.add(h)
                new.append({"id": h, "text": piece, "course": course, "source": f.name, "loc": loc})
        if not new:
            report.append(f"⚠️ {f.name}: no new text found (empty, scanned image, or already indexed)")
            continue
        for i in range(0, len(new), 128):
            batch = new[i:i + 128]
            embs = embedder.encode(["passage: " + c["text"] for c in batch],
                                   normalize_embeddings=True, show_progress_bar=False).tolist()
            collection.add(ids=[c["id"] for c in batch], documents=[c["text"] for c in batch], embeddings=embs,
                           metadatas=[{"course": c["course"], "source": c["source"], "loc": c["loc"]} for c in batch])
        chunks.extend(new)
        total += len(new)
        report.append(f"✅ {f.name}: {len(new)} new chunks -> course {course}")
    COURSES[:] = sorted({c["course"] for c in chunks})
    return total, report

# Quick test with a tiny document the model could not know
demo = UPLOAD_DIR / "demo_leave_policy.txt"
demo.write_text("Acme Company Leave Policy\n\nEvery full-time employee receives 21 days of paid annual leave per year. "
                "Unused leave can be carried over for a maximum of 5 days into the next year. "
                "Sick leave requires a medical certificate after 2 consecutive days.\n", encoding="utf-8")
n, rep = ingest_files([demo], course="DEMO")
print("\n".join(rep), "| total chunks in KB:", collection.count())
show_answer(answer_question("How many days of annual leave does a full-time employee get?"))

✅ demo_leave_policy.txt: 1 new chunks -> course DEMO | total chunks in KB: 297
Q: How many days of annual leave does a full-time employee get?
Status: VERIFIED  |  searches: ['How many days of annual leave does a full-time employee get?']

Answer:
A full-time employee receives 21 days of paid annual leave per year [1].

Sources:
  - DEMO | demo_leave_policy.txt | text file  (relevance 1.0)

----------------------------------------------------------------------



# Part 6: Evaluation (does the system refuse when it should?)
8 questions: 5 that the documents can answer, 3 that they cannot. A correct result means: **answered when answerable, `not_found` when not**. This measures the hallucination control of the pipeline.

In [19]:
from IPython.display import display

eval_set = [
    ("What is symmetric encryption?", True),
    ("What is the internet?", True),
    ("What is cloud computing?", True),
    ("What is Docker?", True),
    ("ما هو التشفير المتماثل؟", True),
    ("Who won the 2022 football World Cup?", False),
    ("What is the capital of Australia?", False),
    ("How do I bake a chocolate cake?", False),
]
rows = []
for q, answerable in eval_set:
    res = answer_question(q)
    answered = res["status"] in ("verified", "partial")
    rows.append({"question": q, "should_answer": answerable, "status": res["status"],
                 "rewrites": len(res["trace"]) - 1, "sources": len(res["sources"]),
                 "correct": answered == answerable})
    time.sleep(1)

df_eval = pd.DataFrame(rows)
display(df_eval)
ans_ok = df_eval[df_eval.should_answer].correct.mean()
ref_ok = df_eval[~df_eval.should_answer].correct.mean()
print(f"Answerable questions answered: {ans_ok:.0%}  |  Unanswerable questions refused: {ref_ok:.0%}  |  overall: {df_eval.correct.mean():.0%}")
print(f"Questions that needed a query rewrite: {(df_eval.rewrites > 0).sum()} of {len(df_eval)}")

,question,should_answer,status,rewrites,sources,correct
0,What is symmetric encryption?,True,verified,0,2,True
1,What is the internet?,True,verified,0,4,True
2,What is cloud computing?,True,verified,0,3,True
3,What is Docker?,True,verified,0,2,True
4,ما هو التشفير المتماثل؟,True,verified,0,1,True
5,Who won the 2022 football World Cup?,False,not_found,2,0,True
6,What is the capital of Australia?,False,not_found,2,0,True
7,How do I bake a chocolate cake?,False,not_found,2,0,True


Answerable questions answered: 100%  |  Unanswerable questions refused: 100%  |  overall: 100%
Questions that needed a query rewrite: 3 of 8


# Part 7: Gradio app
Two tabs: **Ask** (answer + sources + the correction trace: which queries were tried and how each was graded) and **Upload documents** (runs the ingestion pipeline on new files, then they are searchable immediately).
`share=True` gives a public link while the Colab session is running.

In [ ]:
import gradio as gr

ICON = {"correct": "✅", "ambiguous": "🟡", "incorrect": "❌"}
BADGE = {"verified": "✅ **Verified**: answered from relevant chunks only",
         "partial": "🟡 **Low confidence**: documents only partly cover this question",
         "not_found": "ℹ️ **Outside the course material**"}

def _fmt_trace(trace):
    out = []
    for i, t in enumerate(trace):
        label = "Original query" if i == 0 else f"Rewrite {i}"
        out.append(f"{ICON.get(t['status'], '•')} **{label}**: `{t['query']}` → {t['status']}")
    return "\n\n".join(out)

def _fmt_sources(sources):
    if not sources:
        return "_No sources_"
    return "\n".join(f"- **{s['course']}** | {s['source']} | {s['loc']} (relevance {s['relevance']})" for s in sources)

def ui_ask(question, course_choice, k):
    if not question or not question.strip():
        return "Please type a question.", "", ""
    course = None if course_choice == "All courses" else course_choice
    try:
        res = answer_question(question, k=int(k), course=course)
    except Exception as e:
        return f"⚠️ Error: {e.__class__.__name__}: {str(e)[:300]}", "", ""
    return f"{BADGE[res['status']]}\n\n{res['answer']}", _fmt_sources(res["sources"]), _fmt_trace(res["trace"])

def ui_upload(files, course_name):
    if not files:
        return "Please choose at least one file.", gr.update()
    paths = [getattr(f, "name", f) for f in files]
    added, report = ingest_files(paths, course_name)
    msg = f"Added **{added}** new chunks. The knowledge base now has **{collection.count()}** chunks.\n\n" + "\n\n".join(report)
    return msg, gr.update(choices=["All courses"] + COURSES)

EXAMPLES = [["What is symmetric encryption?", "All courses", 5],
            ["ما هو التشفير المتماثل؟", "All courses", 5],
            ["What is cloud computing?", "All courses", 5],
            ["Who won the 2022 football World Cup?", "All courses", 5]]

with gr.Blocks(title="Corrective RAG") as demo:
    gr.Markdown("# 🔎 Corrective RAG: Document Q&A\n"
                "Retrieves chunks, **grades** them, **rewrites the query** and retrieves again when the context is weak, "
                "and answers **only from verified context**, with sources.")
    with gr.Tabs():
        with gr.Tab("Ask"):
            with gr.Row():
                with gr.Column(scale=2):
                    q = gr.Textbox(label="Your question (English or Arabic)", lines=3)
                    with gr.Row():
                        course_dd = gr.Dropdown(["All courses"] + COURSES, value="All courses", label="Course filter")
                        k_sl = gr.Slider(3, 8, value=5, step=1, label="Chunks to retrieve (k)")
                    btn = gr.Button("Ask", variant="primary")
                    gr.Examples(EXAMPLES, inputs=[q, course_dd, k_sl])
                with gr.Column(scale=3):
                    ans = gr.Markdown(label="Answer")
                    with gr.Accordion("Sources", open=True):
                        src_md = gr.Markdown()
                    with gr.Accordion("Correction trace (queries tried)", open=True):
                        trace_md = gr.Markdown()
            btn.click(ui_ask, [q, course_dd, k_sl], [ans, src_md, trace_md])
            q.submit(ui_ask, [q, course_dd, k_sl], [ans, src_md, trace_md])
        with gr.Tab("Upload documents"):
            gr.Markdown("Add **PDF, DOCX, PPTX, TXT or CSV** files. They are extracted, cleaned, split, embedded and stored, then you can ask about them right away.")
            up_files = gr.File(file_count="multiple", type="filepath", label="Documents")
            up_course = gr.Textbox(value="UPLOADS", label="Course / collection name for these files")
            up_btn = gr.Button("Add to knowledge base", variant="primary")
            up_msg = gr.Markdown()
            up_btn.click(ui_upload, [up_files, up_course], [up_msg, course_dd])

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a68a6a1f7ce3540232.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Deploying permanently (Hugging Face Spaces)
1. Copy the code cells (without the `!pip` line, the demo/test cells and the evaluation cell) into `app.py`. Keep the Gradio block last.
2. Build the index once at startup from your `data/` folder (or commit `chroma_db/`).
3. `requirements.txt`: `pypdf python-docx python-pptx chromadb sentence-transformers langchain-text-splitters pandas numpy gradio groq`
4. In the Space settings add `GROQ_API_KEY` as a **Secret** (never put the key in the code).
5. Create a new Space with the **Gradio** SDK and push the files.

Note: on a free Space the disk is temporary, so documents uploaded through the app disappear when the Space restarts. The course documents are rebuilt at startup.